# Final Grand Baseline: Algebraic Hashing vs. State-of-the-Art
This notebook provides a clean, streamlined benchmark execution pipeline:
- Evaluates all 10 benchmark datasets (`cora`, `citeseer`, `pubmed`, `wikics`, `chameleon`, `squirrel`, `photo`, `computers`, `amazon-ratings`, `dblp`).
- Compares proposed algebraic hashing (`NyS-Binary` ) against state-of-the-art baselines (`node2binary`, `NodeSig`, `NodeSketch`, `Bi-GCN`, `Raw Features`).
- Evaluates across 9 downstream classifiers over 3 random seeds.


Alg_V1_Nystrom_Blend and Nys_Binary are same 

In [ ]:
# ============================================================
# 0. IMPORTS & ENVIRONMENT SETUP
# ============================================================
import os, sys, time, warnings, importlib
warnings.filterwarnings('ignore')

# Force the TESTS/Github directory to be first in sys.path
BENCHMARK_DIR = r'./'
if BENCHMARK_DIR in sys.path:
    sys.path.remove(BENCHMARK_DIR)
sys.path.insert(0, BENCHMARK_DIR)

import torch
import numpy as np
import pandas as pd

for m in list(sys.modules.keys()):
    if m == 'torch_geometric' and not hasattr(sys.modules[m], 'typing'):
        del sys.modules[m]
try:
    import torch_geometric
    import torch_geometric.typing
except ImportError:
    pass

for mod in ['advanced_baselines', 'benchmark_new_algebraic']:
    if mod in sys.modules:
        del sys.modules[mod]

import advanced_baselines
import benchmark_new_algebraic

print(f'advanced_baselines loaded from: {advanced_baselines.__file__}')
print(f'Has fuse_alg_v1_randomized_blend: {hasattr(advanced_baselines, "fuse_alg_v1_randomized_blend")}')
print(f'Has fuse_alg_v1_random_blend: {hasattr(advanced_baselines, "fuse_alg_v1_random_blend")}')
print(f'benchmark_new_algebraic loaded from: {benchmark_new_algebraic.__file__}')

load_dataset = benchmark_new_algebraic.load_dataset
extract_features = benchmark_new_algebraic.extract_features
run_benchmark = benchmark_new_algebraic.run_benchmark
print_summary = benchmark_new_algebraic.print_summary
CLASSIFIER_REGISTRY = benchmark_new_algebraic.CLASSIFIER_REGISTRY
SEEDS = benchmark_new_algebraic.SEEDS
DEVICE = benchmark_new_algebraic.DEVICE

print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
print(f'Available classifiers ({len(CLASSIFIER_REGISTRY)}): {list(CLASSIFIER_REGISTRY.keys())}')


advanced_baselines loaded from: ./\advanced_baselines.py
Has fuse_alg_v1_randomized_blend: True
Has fuse_alg_v1_random_blend: True
benchmark_new_algebraic loaded from: ./\benchmark_new_algebraic.py
Device: cuda
PyTorch version: 2.5.1
Available classifiers (10): ['SVM', 'LogisticRegression', 'MLP', 'GCN', 'GraphSAGE', 'LINKX', 'SADP', 'SADP_Two_Way', 'SADP_Surrogate', 'H2GCN']


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

load_previous_checkpoint = True  # Set to True or False

load_cached_binary = True  # Set to True or False

# --- Training Label Ratio ---
train_ratio = 0.70

# --- Datasets ---
datasets = [
    'cora',
    'citeseer',
    'pubmed',
    'wikics',
    'chameleon',
    'squirrel',
    'photo',
    'computers',
    'amazon-ratings',
    'dblp',


    ##### Highly Heterophilic Datasets #####
    # 'actor',
    # 'texas',
    # 'winconsin',
    # 'minesweeper',

    ### Large Datasets ###
    # 'ogbn-arxiv',
    # 'ogbn-products'
]

# --- Embedding Methods ---
methods = [
    #  External baselines 
    'node2binary',              # Official WWW 2025 Leiden random-walk binary embeddings 
    'NodeSig',                  # Official IEEE/ACM ASONAM 2022 binary random-walk signatures 
    'NodeSketch',               # Official ACM SIGKDD 2019 min-wise hashing binary embeddings (modulo 2 fix) 
    'Bi-GCN',                   # Official CVPR 2021 binary GCN trained 
    
    # Internal baselines
    'Raw Features',
    'Raw Features Bin',
    'Top-K CE Adjacency',
    
    #  Proposed Algebraic Methods
    'Alg_V1_Nystrom_Blend',        # Official Proposed ALgorithm Nys-Binary
    'Nys_Binary_Unsupervised',
    # 'Nys_Binary_1pct',
    # 'Nys_Binary_10pct'

    # Other version of the proposed Methods
    # 'Alg_V1_SVD_Blend',         
    # 'Alg_V1_Randomized_Blend',  
    # 'Alg_V1_Chebyshev_Blend',  
]

# --- Downstream Classifiers (10 Models) ---
classifiers = {
    'LogisticRegression': CLASSIFIER_REGISTRY['LogisticRegression'],
    'MLP':          CLASSIFIER_REGISTRY['MLP'],
    'GCN':          CLASSIFIER_REGISTRY['GCN'],
    'GraphSAGE':    CLASSIFIER_REGISTRY['GraphSAGE'],
    'LINKX':        CLASSIFIER_REGISTRY['LINKX'],
    'SADP':         CLASSIFIER_REGISTRY['SADP'],
    'SADP_Two_Way': CLASSIFIER_REGISTRY['SADP_Two_Way'],
    'SADP_Surrogate': CLASSIFIER_REGISTRY['SADP_Surrogate'],
    'H2GCN':        CLASSIFIER_REGISTRY['H2GCN'],
}

# --- Random Seeds (3 seeds for statistical significance) ---
seeds = [42, 123, 77]

# --- Embedding Dimension ---
emb_dim = 250

# --- Device ---
device = DEVICE

# --- Print Configuration ---
print('=' * 70)
print('  BENCHMARK CONFIGURATION')
print('=' * 70)
print(f'Train Ratio:   {train_ratio:.3f} ({int(round(train_ratio*100))}% labels)')
print(f'Datasets ({len(datasets)}):    {datasets}')
print(f'Methods  ({len(methods)}):     {methods}')
print(f'Classifiers ({len(classifiers)}): {list(classifiers.keys())}')
print(f'Seeds    ({len(seeds)}):       {seeds}')
print(f'Emb Dim:       {emb_dim}')
print(f'Device:        {device}')
print(f'Load Checkpoint (✓/✗): {load_previous_checkpoint}')
print(f'Load Cached Binary: {load_cached_binary}')
print('=' * 70)


  BENCHMARK CONFIGURATION
Train Ratio:   0.700 (70% labels)
Datasets (10):    ['cora', 'citeseer', 'pubmed', 'wikics', 'chameleon', 'squirrel', 'photo', 'computers', 'amazon-ratings', 'dblp']
Methods  (2):     ['Nys_Binary_1pct', 'Nys_Binary_10pct']
Classifiers (9): ['LogisticRegression', 'MLP', 'GCN', 'GraphSAGE', 'LINKX', 'SADP', 'SADP_Two_Way', 'SADP_Surrogate', 'H2GCN']
Seeds    (3):       [42, 123, 77]
Emb Dim:       250
Device:        cuda
Load Checkpoint (✓/✗): True
Load Cached Binary: True


In [ ]:
# ============================================================
# RUN THE FULL BENCHMARK
# ============================================================

print('Starting Full Benchmark...')
print('=' * 70)

df = run_benchmark(
    datasets=datasets,
    methods=methods,
    classifiers=classifiers,
    seeds=seeds,
    emb_dim=emb_dim,
    device_str=device,
    skip_existing=load_previous_checkpoint,
    train_ratio=train_ratio,
    load_cached_binary=load_cached_binary,
)

print('\n' + '=' * 70)
print('  BENCHMARK COMPLETE!')
print('=' * 70)


Starting Full Benchmark...
[Checkpoint Engine] Loaded 9710 existing evaluations from 730 CSV checkpoint files.

  Dataset: cora

  Seed: 42 | Device: cuda  (train=1895, test=813)
    Nys_Binary_1pct         [Skipped - Checkpoint hit: 9 classifiers loaded]
    Nys_Binary_10pct        gen=   0.10s  shape=(2708, 250)
    Geometry                D_intra=74.3759  D_inter=115.2628 R=1.5497  
      [LogisticRegression  ]  Acc= 76.75%  F1m= 75.33%  train= 0.39s
      [MLP                 ]  Acc= 78.84%  F1m= 77.61%  train= 0.42s
      [GCN                 ]  Acc= 83.52%  F1m= 83.12%  train= 0.87s
      [GraphSAGE           ]  Acc= 80.81%  F1m= 80.62%  train= 0.66s
      [LINKX               ]  Acc= 78.11%  F1m= 76.63%  train= 1.14s
      [SADP                ]  Acc= 54.49%  F1m= 46.03%  train=30.93s
      [SADP_Two_Way        ]  Acc= 80.21%  F1m= 77.10%  train=26.86s
      [SADP_Surrogate      ]  Acc= 76.01%  F1m= 74.16%  train=48.03s
      [H2GCN               ]  Acc= 82.90%  F1m= 82.41%  tra

In [ ]:
# ============================================================
# SAVE RESULTS TO CSV
# ============================================================
import os
import pandas as pd
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = os.path.join(BENCHMARK_DIR, "Results")
os.makedirs(results_dir, exist_ok=True)

# Save timestamped result file
pct_tag = f"{int(round(train_ratio * 100))}pct"
results_path = os.path.join(results_dir, f"final_benchmark_results_{pct_tag}_{timestamp}.csv")
df.to_csv(results_path, index=False)
print(f"Results saved to: {results_path}")

# Also save latest consolidated results
latest_path = os.path.join(results_dir, "final_benchmark_results_latest.csv")
df.to_csv(latest_path, index=False)
print(f"Latest results saved to: {latest_path}")


Results saved to: ./\Results\final_benchmark_results_70pct_20260924_011258.csv
Latest results saved to: ./\Results\final_benchmark_results_latest.csv


In [ ]:
# ============================================================
# DATASET-BY-DATASET SUMMARY REPORT
# ============================================================
import importlib
import benchmark_new_algebraic
import advanced_baselines
importlib.reload(advanced_baselines)
importlib.reload(benchmark_new_algebraic)
from benchmark_new_algebraic import print_summary

print_summary(df)



  DATASET: CORA (Mean ± Std % across seeds)
classifier                 GCN     GraphSAGE         H2GCN         LINKX LogisticRegression           MLP          SADP SADP_Surrogate  SADP_Two_Way
method                                                                                                                                              
Nys_Binary_10pct  83.93 ± 0.38  82.00 ± 1.17  82.29 ± 0.53  78.48 ± 0.53       79.74 ± 2.61  80.48 ± 1.51  63.59 ± 7.93   78.11 ± 1.83  86.27 ± 5.28
Nys_Binary_1pct   82.45 ± 1.25  81.67 ± 1.86  80.24 ± 1.44  73.84 ± 1.48       76.10 ± 2.43  74.99 ± 3.97  49.00 ± 8.41   62.61 ± 5.99  75.13 ± 1.22

  DATASET: CITESEER (Mean ± Std % across seeds)
classifier                 GCN     GraphSAGE         H2GCN         LINKX LogisticRegression           MLP          SADP SADP_Surrogate  SADP_Two_Way
method                                                                                                                                              
Nys_Binary_1

In [ ]:
# ============================================================
# OVERALL AVERAGE ACCURACY & RUNTIME TABLES (MEAN ± STD)
# ============================================================
if 'method' in df.columns and 'classifier' in df.columns and 'accuracy' in df.columns:
    m = df.pivot_table(values='accuracy', index='method', columns='classifier', aggfunc='mean')
    s = df.pivot_table(values='accuracy', index='method', columns='classifier', aggfunc='std', dropna=False).fillna(0.0)
    
    overall_combined = pd.DataFrame(index=m.index, columns=m.columns)
    for col in m.columns:
        overall_combined[col] = [f"{mv:.2f} ± {sv:.2f}" for mv, sv in zip(m[col], s[col])]

    print("=" * 80)
    print("  OVERALL AVERAGE ACCURACY (Mean ± Std % across all datasets & seeds)")
    print("=" * 80)
    print(overall_combined.to_string())
    print()

    if 'gen_time' in df.columns:
        print("=" * 80)
        print("  OVERALL EMBEDDING GENERATION TIME (Mean ± Std seconds across all runs)")
        print("=" * 80)
        gen_df = df.groupby(['dataset', 'method', 'seed'])['gen_time'].first().reset_index()
        t_stats = gen_df.groupby('method')['gen_time'].agg(
            Mean_s='mean',
            Std_s='std',
            Median_s='median'
        ).fillna(0.0)
        time_table = pd.DataFrame(index=t_stats.index)
        time_table['Generation Time (Mean ± Std)'] = [
            f"{r['Mean_s']:>8.4f}s ± {r['Std_s']:>8.4f}s" for _, r in t_stats.iterrows()
        ]
        time_table['Median (s)'] = [f"{r['Median_s']:>8.4f}s" for _, r in t_stats.iterrows()]
        print(time_table.to_string())


  OVERALL AVERAGE ACCURACY (Mean ± Std % across all datasets & seeds)
classifier                  GCN      GraphSAGE          H2GCN          LINKX LogisticRegression            MLP           SADP SADP_Surrogate   SADP_Two_Way
method                                                                                                                                                     
Nys_Binary_10pct  72.57 ± 19.17  73.43 ± 17.57  74.32 ± 16.72  76.95 ± 12.59      70.89 ± 18.61  73.16 ± 16.62  58.31 ± 20.80  64.89 ± 23.93  80.86 ± 14.16
Nys_Binary_1pct   71.50 ± 19.11  72.24 ± 17.83  73.43 ± 16.68  75.53 ± 12.85      68.36 ± 18.29  70.79 ± 16.70  51.50 ± 19.87  60.77 ± 22.02  76.59 ± 13.85

  OVERALL EMBEDDING GENERATION TIME (Mean ± Std seconds across all runs)
                 Generation Time (Mean ± Std) Median (s)
method                                                  
Nys_Binary_10pct          0.2807s ±   0.2129s    0.2774s
Nys_Binary_1pct           0.3767s ±   0.3727s    0.3018s


In [ ]:
# ============================================================
# MEAN GENERATION TIME ACROSS DATASETS (MEAN ± STD)
# ============================================================
work_df = df.copy()
col_map = {
    'Dataset': 'dataset',
    'Method': 'method',
    'Seed': 'seed',
    'Gen_Time_s': 'gen_time',
    'gen_time_s': 'gen_time',
    'GenTime': 'gen_time',
}
work_df.rename(columns={k: v for k, v in col_map.items() if k in work_df.columns}, inplace=True)

if 'gen_time' in work_df.columns and 'dataset' in work_df.columns and 'method' in work_df.columns:
    gen_df = work_df.groupby(['dataset', 'method', 'seed'])['gen_time'].first().reset_index()

    # ---  Mean Time Across Datasets (Mean ± Std for all datasets) ---
    ds_means = gen_df.groupby(['method', 'dataset'])['gen_time'].mean().reset_index()
    across_ds = ds_means.groupby('method')['gen_time'].agg(
        Mean_s='mean',
        Std_s='std',
        Median_s='median',
        Min_s='min',
        Max_s='max'
    ).fillna(0.0).sort_values(by='Mean_s', ascending=True)

    summary_table = pd.DataFrame(index=across_ds.index)
    summary_table['Generation Time (Mean ± Std)'] = [
        f"{r['Mean_s']:>8.4f}s ± {r['Std_s']:>8.4f}s" for _, r in across_ds.iterrows()
    ]
    summary_table['Median (s)'] = [f"{r['Median_s']:>8.4f}s" for _, r in across_ds.iterrows()]
    summary_table['Min (s)'] = [f"{r['Min_s']:>8.4f}s" for _, r in across_ds.iterrows()]
    summary_table['Max (s)'] = [f"{r['Max_s']:>8.4f}s" for _, r in across_ds.iterrows()]

    print('=' * 80)
    print('  MEAN EMBEDDING GENERATION TIME ACROSS DATASETS (Mean ± Std for all datasets)')
    print('=' * 80)
    print(summary_table.to_string())
    print()

    # --- Dataset-by-Dataset Breakdown (Mean ± Std across seeds per dataset) ---
    p_mean = gen_df.pivot_table(values='gen_time', index='method', columns='dataset', aggfunc='mean').reindex(across_ds.index)
    p_std = gen_df.pivot_table(values='gen_time', index='method', columns='dataset', aggfunc='std').fillna(0.0).reindex(across_ds.index)

    time_per_dataset = pd.DataFrame(index=p_mean.index, columns=p_mean.columns)
    for col in p_mean.columns:
        time_per_dataset[col] = [f"{m:.4f}s ± {s:.4f}s" for m, s in zip(p_mean[col], p_std[col])]

    print('=' * 80)
    print('  DATASET-BY-DATASET GENERATION TIME BREAKDOWN (Mean ± Std seconds across seeds)')
    print('=' * 80)
    print(time_per_dataset.to_string())


  MEAN EMBEDDING GENERATION TIME ACROSS DATASETS (Mean ± Std for all datasets)
                 Generation Time (Mean ± Std) Median (s)    Min (s)    Max (s)
method                                                                        
Nys_Binary_10pct          0.2807s ±   0.2182s    0.2655s    0.0362s    0.6380s
Nys_Binary_1pct           0.3767s ±   0.3187s    0.2674s    0.0315s    1.0002s

  DATASET-BY-DATASET GENERATION TIME BREAKDOWN (Mean ± Std seconds across seeds)
dataset              amazon-ratings          chameleon           citeseer          computers               cora               dblp              photo             pubmed           squirrel             wikics
method                                                                                                                                                                                                        
Nys_Binary_10pct  0.2917s ± 0.0264s  0.0821s ± 0.0031s  0.0362s ± 0.0041s  0.6380s ± 0.0709s  0.0548s ± 0.037